In [1]:
import numpy as np
from scipy.optimize import minimize
from eval_utils import get_iou

def generate_utility_functions(num_agents, subissues_per_issue, total_utility, target_iou):
    """
    Generate utility functions for agents with constraints.
    """
    # Initialize agents with random scores
    agents = {
        f"agent_{i}": {
            "scores": {f"issue_{j}": np.random.rand(s) for j, s in enumerate(subissues_per_issue)},
        }
        for i in range(num_agents)
    }
    
    for agent in agents.values():
        agent["scores"]["min"]= np.random.normal(0.5, 0.1, 1)

    # Normalize initial scores to meet the total utility constraint
    for agent in agents.values():
        total = sum(np.sum(scores) for scores in agent["scores"].values())
        for issue, scores in agent["scores"].items():
            agent["scores"][issue] = (scores / total) * total_utility

    # Objective: Minimize deviation from the target IoU
    def objective(flat_scores):
        # Reshape flattened scores into agents' utility functions
        idx = 0
        for agent in agents.values():
            for issue, scores in agent["scores"].items():
                size = len(scores)
                agent["scores"][issue] = flat_scores[idx:idx + size]
                idx += size
        # Compute IoU
        avg_iou = get_iou(agents, use_numpy=True)
        return abs(avg_iou - target_iou)

    # Flatten the agents' utility scores for optimization
    flat_scores = np.concatenate([scores for agent in agents.values() for scores in agent["scores"].values()])

    # Constraints: Total utility per agent must equal total_utility, and all scores must be positive
    constraints = []
    idx = 0
    for agent in agents.values():
        max_sum_constraint = {
            'type': 'eq',
            'fun': lambda x, idx=idx, agent=agent: sum(
                np.max(x[idx + sum(len(scores) for scores in list(agent["scores"].values())[:i]): 
                        idx + sum(len(scores) for scores in list(agent["scores"].values())[:i + 1])]) 
                for i in range(len(agent["scores"]))
            ) - 100
        }
        constraints.append(max_sum_constraint)
        idx += sum(len(scores) for scores in agent["scores"].values())

    # Non-negativity constraint
    bounds = [(0, None) for _ in range(len(flat_scores))]

    # Perform optimization
    result = minimize(
        objective,
        flat_scores,
        constraints=constraints,
        bounds=bounds,
        method='SLSQP',
        options={'maxiter': 1000, 'disp': True}
    )

    # Update agents with optimized scores
    idx = 0
    for agent in agents.values():
        for issue, scores in agent["scores"].items():
            size = len(scores)
            agent["scores"][issue] = result.x[idx:idx + size]
            idx += size

    return agents

In [ ]:


# Example Usage
num_agents = 6
subissues_per_issue = [3, 3, 4, 4, 5]
total_utility = 100
target_iou = 0.5

agents = generate_utility_functions(num_agents, subissues_per_issue, total_utility, target_iou)

# Display Results
for agent_name, agent_data in agents.items():
    print(f"{agent_name}:")
    for issue, scores in agent_data["scores"].items():
        print(f"  {issue}: {np.floor(scores)}")



In [15]:
def check_sum_of_utilities(agents, total_utility):
    """
    Check if the sum of utilities for each agent equals the total utility.
    """
    for agent_name, agent_data in agents.items():
        total = sum(np.sum(np.max(scores)) for scores in agent_data["scores"].values())
        if not np.isclose(total, total_utility, atol=1e-6):
            print(f"Sum of utilities for {agent_name} is incorrect: {total} != {total_utility}")
            return False
    print("Sum of utilities for all agents is correct.")
    return True

def check_iou(agents, target_iou, tolerance=0.01):
    """
    Check if the IoU of the agents' scores is close to the target.
    """
    avg_iou = get_iou(agents, use_numpy=True)

    if abs(avg_iou - target_iou) > tolerance:
        print(f"Average IoU is outside tolerance: {avg_iou} != {target_iou}")
        return False
    print(f"Average IoU is within tolerance: {avg_iou} ≈ {target_iou}")
    return True

In [ ]:
check_sum_of_utilities(agents, total_utility)
check_iou(agents, target_iou)